# Schema Evolution And Migrations

## Quick notes
### What this covers
- Changing a table after it has data: `ALTER TABLE` ADD / RENAME / DROP COLUMN, and what SQLite refuses to do.
- The official 12-step table rebuild for everything `ALTER TABLE` cannot do, with foreign keys handled safely.
- Safe change patterns (expand / contract), adding NOT NULL columns, renaming tables, data migrations.
- Tracking versions (`PRAGMA user_version`, a `schema_migrations` table), a tiny migration runner, schema checksums and diffs.
- Rollback, zero-downtime thinking and testing migrations.
### Key points
- `ALTER TABLE` in SQLite can: `RENAME TO`, `RENAME COLUMN`, `ADD COLUMN`, `DROP COLUMN`. Nothing else (no change of type, default, constraints).
- Anything else = rebuild: create new table, copy rows, drop old, rename new.
- DDL is transactional in SQLite, so a migration inside `BEGIN ... COMMIT` is all-or-nothing.
- A migration is a numbered, never-edited-after-release SQL step. Record which ones ran, so running twice is safe.
- `PRAGMA user_version` is a free integer in the file header: perfect for "which schema version is this database at".
- Prefer small backward-compatible steps (expand, migrate, contract) over one big-bang change.
### Gotchas (the part worth revising twice)
- `ADD COLUMN ... NOT NULL` needs a non-NULL constant `DEFAULT`; `UNIQUE`, `PRIMARY KEY` and `DEFAULT CURRENT_TIMESTAMP` columns cannot be added.
- `DROP COLUMN` fails if the column is indexed, a key, used by a view/trigger/CHECK, or referenced by a foreign key.
- `PRAGMA foreign_keys` is a silent no-op inside a transaction: switch it OFF before `BEGIN`, and ON again after `COMMIT`.
- Rebuilding the parent of a foreign key with foreign keys ON fails at `DROP TABLE` (the drop acts like a delete of every row).
- Rename the NEW table into the old name; never rename the old table away first, because views and triggers follow the renamed table.
- A migrated database and a freshly created one can have different `sqlite_master.sql` text for the same structure.
- Never edit a migration that has already run somewhere. Add a new one.
### When it matters
Every app that lives longer than its first release: new features need new columns, bad designs need fixing, and the data is already in production.
### One-line recap example
```sql
ALTER TABLE users ADD COLUMN plan TEXT NOT NULL DEFAULT 'free';   -- safe, instant
PRAGMA user_version = 3;                                          -- remember where we are
```

## 01. Seed data
- A tiny shop: `customers`, `orders`, an index and a view over both.
- Every later section changes this schema, so the data stays small on purpose.
- Dates are plain fixed text; the migration examples never depend on the clock.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    email       TEXT
);
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
    total_cents INTEGER NOT NULL,
    status      TEXT NOT NULL
);
CREATE INDEX idx_orders_customer ON orders(customer_id);
CREATE VIEW v_order_summary AS
    SELECT o.order_id, c.name AS customer, o.status
    FROM orders o JOIN customers c ON c.customer_id = o.customer_id;
INSERT INTO customers VALUES (1,'Asha Rao','asha@example.com'),(2,'Ben Carter','ben@example.com'),(3,'Chen Wei',NULL);
INSERT INTO orders VALUES (101,1,2500,'paid'),(102,1,9900,'shipped'),(103,2,1200,'paid'),(104,3,15000,'new'),(105,2,700,'cancelled');
SELECT order_id, customer, status FROM v_order_summary ORDER BY order_id;

order_id | customer   | status   
---------+------------+----------
101      | Asha Rao   | paid     
102      | Asha Rao   | shipped  
103      | Ben Carter | paid     
104      | Chen Wei   | new      
105      | Ben Carter | cancelled
(5 rows)



## 02. ADD COLUMN: the easy, safe change
- `ALTER TABLE ... ADD COLUMN` adds the column at the end. Existing rows get the default (or NULL).
- It is cheap: SQLite does not rewrite the table, it just records the new column and the default in the schema.
- A constant `DEFAULT` plus `NOT NULL` is fine: old rows read the default.
- Column constraints like `CHECK` and `REFERENCES` are allowed (a `REFERENCES` column with foreign keys ON must default to NULL).

In [3]:
%%sql
ALTER TABLE customers ADD COLUMN country TEXT;
ALTER TABLE customers ADD COLUMN tier TEXT NOT NULL DEFAULT 'basic';
SELECT customer_id, name, country, tier FROM customers ORDER BY customer_id;

customer_id | name       | country | tier 
------------+------------+---------+------
1           | Asha Rao   | NULL    | basic
2           | Ben Carter | NULL    | basic
3           | Chen Wei   | NULL    | basic
(3 rows)



## 03. What ADD COLUMN refuses to do
- `NOT NULL` without a default: old rows would have nothing valid to hold.
- `UNIQUE` or `PRIMARY KEY`: adding either needs an index build, which `ADD COLUMN` cannot do.
- A non-constant default such as `CURRENT_TIMESTAMP`: SQLite cannot compute it for rows that already exist.
- Workarounds: add the plain column, backfill, then add a separate `CREATE UNIQUE INDEX`; or rebuild the table (section 06).
- Each error below leaves the schema untouched.

In [4]:
%%sql expect_error
ALTER TABLE customers ADD COLUMN note TEXT NOT NULL;
ALTER TABLE customers ADD COLUMN handle TEXT UNIQUE;
ALTER TABLE customers ADD COLUMN created TEXT DEFAULT CURRENT_TIMESTAMP;
ALTER TABLE customers ADD COLUMN extra_id INTEGER PRIMARY KEY;

ERROR: Cannot add a NOT NULL column with default value NULL

ERROR: Cannot add a UNIQUE column

ERROR: Cannot add a column with non-constant default

ERROR: Cannot add a PRIMARY KEY column



## 04. RENAME COLUMN
- `ALTER TABLE t RENAME COLUMN old TO new` rewrites every place that mentions the column: indexes, views, triggers, CHECKs, foreign keys.
- It does not touch your application code or raw SQL strings. Search the code base for the old name.
- Below, the view still works, and its stored text was rewritten to say `full_name`.

In [5]:
%%sql
ALTER TABLE customers RENAME COLUMN name TO full_name;
SELECT order_id, customer, status FROM v_order_summary WHERE order_id <= 102 ORDER BY order_id;
SELECT sql FROM sqlite_master WHERE name = 'v_order_summary';

order_id | customer | status 
---------+----------+--------
101      | Asha Rao | paid   
102      | Asha Rao | shipped
(2 rows)

sql                                                                                                                                                        
-----------------------------------------------------------------------------------------------------------------------------------------------------------
CREATE VIEW v_order_summary AS
    SELECT o.order_id, c.full_name AS customer, o.status
    FROM orders o JOIN customers c ON c.customer_id = o.customer_id
(1 row)



## 05. DROP COLUMN and its limits
- `DROP COLUMN` (SQLite 3.35+) physically rewrites each row without the column.
- It fails when the column is: a PRIMARY KEY, UNIQUE, indexed, used in a CHECK, a foreign key, a view, a trigger, or a generated-column expression.
- The refusal is a feature: SQLite will not silently break something that depends on the column.
- Fix by removing the dependency first (drop the index/view), then drop the column. Or rebuild.

In [6]:
%%sql expect_error
CREATE INDEX idx_customers_tier ON customers(tier);
ALTER TABLE customers DROP COLUMN tier;
ALTER TABLE customers DROP COLUMN customer_id;
ALTER TABLE orders DROP COLUMN customer_id;

ERROR: error in index idx_customers_tier after drop column: no such column: tier

ERROR: cannot drop PRIMARY KEY column: "customer_id"

ERROR: error in index idx_orders_customer after drop column: no such column: customer_id



In [7]:
%%sql
DROP INDEX idx_customers_tier;
ALTER TABLE customers DROP COLUMN tier;
SELECT name FROM pragma_table_info('customers') ORDER BY cid;

name       
-----------
customer_id
full_name  
email      
country    
(4 rows)



## 06. The official 12-step rebuild (the recipe)
- For anything `ALTER TABLE` cannot do (change type, add CHECK / UNIQUE / foreign key, change a default, reorder columns, drop a PK column) SQLite documents one recipe:
1. `PRAGMA foreign_keys = OFF` (before the transaction, it is ignored inside one).
2. `BEGIN` a transaction.
3. Remember the SQL of indexes, triggers and views attached to the table (`SELECT type, sql FROM sqlite_master WHERE tbl_name = 'X'`).
4. `CREATE TABLE new_X (...)` with the new definition.
5. `INSERT INTO new_X SELECT ... FROM X` (transform data here).
6. `DROP TABLE X`.
7. `ALTER TABLE new_X RENAME TO X`.
8. Recreate the indexes, triggers and views from step 3 (drop views first if they block step 7).
9. If views refer to X in an affected way, drop and recreate them.
10. `PRAGMA foreign_key_check` (any row returned is a broken reference).
11. `COMMIT`.
12. `PRAGMA foreign_keys = ON` again.
- Steps 1 and 12 matter most when other tables point at X with a foreign key.

## 07. Rebuild demo: add CHECK and UNIQUE to a parent table
- Goal: `customers.email` must be unique and look like an email. `ALTER TABLE` cannot add either as a constraint.
- `customers` is the parent of `orders`, so this is the hard case.
- First the mistake: do the rebuild with foreign keys still ON. The `DROP TABLE` fails because the drop behaves like deleting every customer while orders still point at them.
- A failed statement leaves the transaction open, so the cell ends with `ROLLBACK`.

In [8]:
%%sql expect_error
BEGIN;
CREATE TABLE customers_new (
    customer_id INTEGER PRIMARY KEY,
    full_name   TEXT NOT NULL,
    email       TEXT UNIQUE CHECK (email IS NULL OR email LIKE '%_@_%'),
    country     TEXT
);
INSERT INTO customers_new SELECT customer_id, full_name, email, country FROM customers;
DROP TABLE customers;
ROLLBACK;
SELECT count(*) AS customers_still_there FROM customers;

ERROR: FOREIGN KEY constraint failed

customers_still_there
---------------------
3                    
(1 row)



## 08. Rebuild demo, done right
- Foreign keys OFF, then `BEGIN`. The view mentions `customers`, and `RENAME` re-checks every view, so we drop the view first and recreate it afterwards.
- `foreign_key_check` returns one row per broken reference. Zero rows means the rebuild kept every link valid.
- Foreign keys go back ON at the end. The `orders` rows kept pointing at `customers` by name, so after the rename they point at the new table.

In [9]:
%%sql
PRAGMA foreign_keys = OFF;
BEGIN;
DROP VIEW v_order_summary;
CREATE TABLE customers_new (
    customer_id INTEGER PRIMARY KEY,
    full_name   TEXT NOT NULL,
    email       TEXT UNIQUE CHECK (email IS NULL OR email LIKE '%_@_%'),
    country     TEXT
);
INSERT INTO customers_new SELECT customer_id, full_name, email, country FROM customers;
DROP TABLE customers;
ALTER TABLE customers_new RENAME TO customers;
CREATE VIEW v_order_summary AS
    SELECT o.order_id, c.full_name AS customer, o.status
    FROM orders o JOIN customers c ON c.customer_id = o.customer_id;
PRAGMA foreign_key_check;
COMMIT;
PRAGMA foreign_keys = ON;
PRAGMA foreign_keys;

table | rowid | parent | fkid
------+-------+--------+-----
(0 rows)

foreign_keys
------------
1           
(1 row)



In [10]:
%%sql expect_error
INSERT INTO customers (customer_id, full_name, email) VALUES (4, 'Dina Lee', 'not-an-email');
INSERT INTO customers (customer_id, full_name, email) VALUES (5, 'Dina Lee', 'asha@example.com');
SELECT order_id, customer FROM v_order_summary ORDER BY order_id LIMIT 2;

ERROR: CHECK constraint failed: email IS NULL OR email LIKE '%_@_%'

ERROR: UNIQUE constraint failed: customers.email

order_id | customer
---------+---------
101      | Asha Rao
102      | Asha Rao
(2 rows)



## 09. foreign_key_check as a safety net
- `PRAGMA foreign_key_check` lists rows whose foreign key points at nothing. Columns: child table, rowid, parent table, which FK.
- With foreign keys OFF you can create an orphan; the check finds it, and you fix it before turning enforcement back on.
- Always run it before `COMMIT` in a rebuild. It is cheap insurance.

In [11]:
%%sql
PRAGMA foreign_keys = OFF;
INSERT INTO orders VALUES (900, 77, 100, 'new');
PRAGMA foreign_key_check;
DELETE FROM orders WHERE order_id = 900;
PRAGMA foreign_key_check;
PRAGMA foreign_keys = ON;

table  | rowid | parent    | fkid
-------+-------+-----------+-----
orders | 900   | customers | 0   
(1 row)

table | rowid | parent | fkid
------+-------+--------+-----
(0 rows)



## 10. Adding a NOT NULL column safely
- Case A: a constant default is right for old rows: `ADD COLUMN currency TEXT NOT NULL DEFAULT 'USD'`. One statement, done.
- Case B: the right value depends on the row (derive it from other data). A single default would be wrong.
- Plan for case B: add the column nullable, backfill with `UPDATE`, verify no NULLs remain, then enforce.
- Enforcement without a rebuild: a pair of triggers that reject NULL on insert and update. A rebuild (section 06) can then make it a real `NOT NULL` later.

In [12]:
%%sql
ALTER TABLE orders ADD COLUMN currency TEXT NOT NULL DEFAULT 'USD';
ALTER TABLE orders ADD COLUMN channel TEXT;
UPDATE orders SET channel = CASE WHEN total_cents >= 10000 THEN 'sales-desk' ELSE 'web' END;
SELECT count(*) AS still_null FROM orders WHERE channel IS NULL;
CREATE TRIGGER trg_orders_channel_ins BEFORE INSERT ON orders WHEN NEW.channel IS NULL
BEGIN SELECT RAISE(ABORT, 'orders.channel must not be NULL'); END;
CREATE TRIGGER trg_orders_channel_upd BEFORE UPDATE OF channel ON orders WHEN NEW.channel IS NULL
BEGIN SELECT RAISE(ABORT, 'orders.channel must not be NULL'); END;
SELECT order_id, total_cents, currency, channel FROM orders ORDER BY order_id;

still_null
----------
0         
(1 row)

order_id | total_cents | currency | channel   
---------+-------------+----------+-----------
101      | 2500        | USD      | web       
102      | 9900        | USD      | web       
103      | 1200        | USD      | web       
104      | 15000       | USD      | sales-desk
105      | 700         | USD      | web       
(5 rows)



In [13]:
%%sql expect_error
INSERT INTO orders (order_id, customer_id, total_cents, status) VALUES (106, 1, 500, 'new');
UPDATE orders SET channel = NULL WHERE order_id = 101;

ERROR: orders.channel must not be NULL

ERROR: orders.channel must not be NULL



## 11. Expand / contract: change without breaking running code
- Never rename or drop something old code still reads. Do it in three releases:
- Expand: add the new column next to the old one, keep both in sync (trigger or app dual-write), backfill old rows.
- Migrate: switch the readers and writers to the new column. Both versions of the app still work.
- Contract: when nothing uses the old column, drop the sync trigger and the old column.
- Example: `total_cents` becomes `amount_minor` (same meaning, a more general name).
- The trigger lets the OLD app, which only knows `total_cents`, keep inserting while the new column fills itself.

In [14]:
%%sql
ALTER TABLE orders ADD COLUMN amount_minor INTEGER;
UPDATE orders SET amount_minor = total_cents WHERE amount_minor IS NULL;
CREATE TRIGGER trg_orders_sync_amount AFTER INSERT ON orders WHEN NEW.amount_minor IS NULL
BEGIN UPDATE orders SET amount_minor = NEW.total_cents WHERE order_id = NEW.order_id; END;
INSERT INTO orders (order_id, customer_id, total_cents, status, channel) VALUES (106, 3, 3300, 'new', 'web');
SELECT order_id, total_cents, amount_minor FROM orders WHERE order_id IN (105, 106) ORDER BY order_id;

order_id | total_cents | amount_minor
---------+-------------+-------------
105      | 700         | 700         
106      | 3300        | 3300        
(2 rows)



### Contract step
- Only after every reader uses `amount_minor`: remove the trigger, then drop the old column.
- `total_cents` is `NOT NULL`, which is fine for `DROP COLUMN`. It is not indexed or referenced, so the drop works.
- New code now has to supply `amount_minor` itself.

In [15]:
%%sql
DROP TRIGGER trg_orders_sync_amount;
ALTER TABLE orders DROP COLUMN total_cents;
SELECT name FROM pragma_table_info('orders') ORDER BY cid;
SELECT order_id, amount_minor FROM orders WHERE order_id >= 104 ORDER BY order_id;

name        
------------
order_id    
customer_id 
status      
currency    
channel     
amount_minor
(6 rows)

order_id | amount_minor
---------+-------------
104      | 15000       
105      | 700         
106      | 3300        
(3 rows)



## 12. Renaming tables: views and triggers follow
- `ALTER TABLE a RENAME TO b` rewrites foreign keys in other tables, views and triggers that mention `a` (SQLite 3.26+ with default settings).
- Below, rename `orders` to `sales_orders` and look at what happened to the view, the child table and the trigger.
- Then the trap: renaming the old table away "to make room" for a new one drags every view and foreign key along with it. That is why the recipe renames the NEW table into place.
- The last query shows the view text now pointing at `orders_old`: that is the bug.

In [16]:
%%sql
CREATE TABLE order_notes (note_id INTEGER PRIMARY KEY, order_id INTEGER NOT NULL REFERENCES orders(order_id), note TEXT);
INSERT INTO order_notes VALUES (1, 101, 'gift wrap');
ALTER TABLE orders RENAME TO sales_orders;
SELECT name, sql FROM sqlite_master WHERE name IN ('v_order_summary', 'order_notes') ORDER BY name;
SELECT name FROM sqlite_master WHERE type = 'trigger' AND tbl_name = 'sales_orders' ORDER BY name;
ALTER TABLE sales_orders RENAME TO orders_old;
SELECT sql FROM sqlite_master WHERE name = 'v_order_summary';
ALTER TABLE orders_old RENAME TO orders;
SELECT order_id, customer FROM v_order_summary ORDER BY order_id LIMIT 2;

name            | sql                                                                                                                                                                
----------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------
order_notes     | CREATE TABLE order_notes (note_id INTEGER PRIMARY KEY, order_id INTEGER NOT NULL REFERENCES "sales_orders"(order_id), note TEXT)                                   
v_order_summary | CREATE VIEW v_order_summary AS
    SELECT o.order_id, c.full_name AS customer, o.status
    FROM "sales_orders" o JOIN customers c ON c.customer_id = o.customer_id
(2 rows)

name                  
----------------------
trg_orders_channel_ins
trg_orders_channel_upd
(2 rows)

sql                                                                                                                                                             

## 13. PRAGMA user_version and schema_version
- `user_version`: a 32-bit integer in the file header that SQLite never touches. Yours to use as the schema version.
- `schema_version`: SQLite's own counter, bumped on every schema change so cached statements know to recompile. Read it for curiosity, do not write it.
- `user_version` is saved with the transaction, so it commits or rolls back together with the DDL of the same migration.
- A new database starts at 0. A simple migration scheme is "run every step whose number is greater than `user_version`".

In [17]:
%%sql
PRAGMA user_version;
PRAGMA user_version = 3;
PRAGMA user_version;

user_version
------------
0           
(1 row)

user_version
------------
3           
(1 row)



In [18]:
before = con.execute("PRAGMA schema_version").fetchone()[0]
con.execute("CREATE TABLE tmp_version_probe (x)")
mid = con.execute("PRAGMA schema_version").fetchone()[0]
con.execute("INSERT INTO tmp_version_probe VALUES (1)")
after_insert = con.execute("PRAGMA schema_version").fetchone()[0]
con.execute("DROP TABLE tmp_version_probe")
print("DDL bumps schema_version:", mid > before)
print("plain INSERT does not:    ", after_insert == mid)
print("user_version untouched by DDL:", con.execute("PRAGMA user_version").fetchone()[0])

DDL bumps schema_version: True
plain INSERT does not:     True
user_version untouched by DDL: 3


## 14. A versioned migration runner (Python)
- A migration is `(version, name, sql)`. Versions are numbers that only ever grow.
- The runner: create `schema_migrations` if missing, skip versions already recorded, run each new one inside its own transaction, record it, and set `user_version`.
- The recorded checksum catches the classic mistake: someone edits an old migration after it already ran elsewhere.
- Idempotent: running it again does nothing and returns an empty list.
- It reuses the notebook's `_statements` helper to split a multi-statement script.

In [19]:
import hashlib

def checksum(sql):
    return hashlib.sha256(sql.encode()).hexdigest()[:12]

def migrate(db, migrations, target=None):
    db.execute("CREATE TABLE IF NOT EXISTS schema_migrations "
               "(version INTEGER PRIMARY KEY, name TEXT NOT NULL, checksum TEXT NOT NULL)")
    applied = {v: c for v, c in db.execute("SELECT version, checksum FROM schema_migrations")}
    ran = []
    for version, name, sql in sorted(migrations):
        if target is not None and version > target:
            break
        if version in applied:
            if applied[version] != checksum(sql):
                raise RuntimeError(f"migration {version} ({name}) was edited after it ran")
            continue
        db.execute("BEGIN")
        try:
            for stmt in _statements(sql):
                db.execute(stmt)
            db.execute("INSERT INTO schema_migrations VALUES (?, ?, ?)", (version, name, checksum(sql)))
            db.execute(f"PRAGMA user_version = {version}")
            db.execute("COMMIT")
        except Exception:
            db.execute("ROLLBACK")
            raise
        ran.append(version)
    return ran

MIGRATIONS = [
    (1, "create_users", "CREATE TABLE users (id INTEGER PRIMARY KEY, email TEXT NOT NULL);"),
    (2, "add_user_name", "ALTER TABLE users ADD COLUMN name TEXT;"),
    (3, "backfill_name_and_index", """
        UPDATE users SET name = substr(email, 1, instr(email, '@') - 1) WHERE name IS NULL;
        CREATE UNIQUE INDEX ux_users_email ON users(email);"""),
]

mdb = sqlite3.connect(":memory:", isolation_level=None)
print("first run, up to v2:", migrate(mdb, MIGRATIONS, target=2))
print("second run, all:    ", migrate(mdb, MIGRATIONS))
print("third run (nothing):", migrate(mdb, MIGRATIONS))
print("user_version:", mdb.execute("PRAGMA user_version").fetchone()[0])
print("recorded:", [r[:2] for r in mdb.execute("SELECT * FROM schema_migrations ORDER BY version")])

first run, up to v2: [1, 2]
second run, all:     [3]
third run (nothing): []
user_version: 3
recorded: [(1, 'create_users'), (2, 'add_user_name'), (3, 'backfill_name_and_index')]


## 15. Runner under stress: failure and tampering
- A migration that fails halfway must leave nothing behind. Here statement 1 succeeds, statement 2 is a typo.
- Because both run inside one transaction, the half-built table disappears and the version is not recorded.
- Editing migration 2 after it ran is detected through the checksum.
- Fix a bad released migration by adding a new migration (4, 5, ...), never by editing history.

In [20]:
bad = MIGRATIONS + [(4, "half_broken", """
    CREATE TABLE half_done (x);
    INSERT INTO no_such_table VALUES (1);""")]
try:
    migrate(mdb, bad)
except sqlite3.Error as exc:
    print("migration 4 failed:", exc)
print("half_done exists?     ", mdb.execute("SELECT count(*) FROM sqlite_master WHERE name = 'half_done'").fetchone()[0] == 1)
print("user_version still:   ", mdb.execute("PRAGMA user_version").fetchone()[0])
print("versions recorded:    ", [r[0] for r in mdb.execute("SELECT version FROM schema_migrations ORDER BY version")])

edited = [MIGRATIONS[0], (2, "add_user_name", "ALTER TABLE users ADD COLUMN display_name TEXT;"), MIGRATIONS[2]]
try:
    migrate(mdb, edited)
except RuntimeError as exc:
    print("tamper check:", exc)

migration 4 failed: no such table: no_such_table
half_done exists?      False
user_version still:    3
versions recorded:     [1, 2, 3]
tamper check: migration 2 (add_user_name) was edited after it ran


## 16. Checksum of the schema (sqlite_master)
- `sqlite_master` holds the `CREATE` text of every table, index, view and trigger. Hash it to get a fingerprint of the whole schema.
- Use it to assert "staging and production have the same schema" with one comparison.
- Normalise whitespace first, and leave out `sqlite_%` internal objects.
- Gotcha: `ALTER TABLE` edits the stored `CREATE` text (for example `RENAME TO` re-writes the table name in quotes), so a migrated table and a freshly created one can differ in text while being the same structure.
- For a structural check, hash `pragma_table_info` output instead (below, same structure, different text).

In [21]:
def text_fingerprint(db):
    rows = db.execute("SELECT type, name, sql FROM sqlite_master "
                      "WHERE name NOT LIKE 'sqlite_%' AND sql IS NOT NULL ORDER BY type, name").fetchall()
    norm = "\n".join(f"{t}|{n}|{' '.join(s.split())}" for t, n, s in rows)
    return hashlib.sha256(norm.encode()).hexdigest()[:12]

def structure_fingerprint(db):
    parts = []
    for (tbl,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name").fetchall():
        for r in db.execute("SELECT name, type, \"notnull\", dflt_value, pk FROM pragma_table_info(?) ORDER BY name", (tbl,)):
            parts.append(f"{tbl}|{r}")
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:12]

fresh = sqlite3.connect(":memory:")
fresh.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, a TEXT, b TEXT)")
aged = sqlite3.connect(":memory:")
aged.execute("CREATE TABLE t_new (id INTEGER PRIMARY KEY, a TEXT, b TEXT)")
aged.execute("ALTER TABLE t_new RENAME TO t")
print("same text fingerprint?     ", text_fingerprint(fresh) == text_fingerprint(aged))
print("same structure fingerprint?", structure_fingerprint(fresh) == structure_fingerprint(aged))
print("stored text of the aged table:", aged.execute("SELECT sql FROM sqlite_master WHERE name = 't'").fetchone()[0])

same text fingerprint?      False
same structure fingerprint? True
stored text of the aged table: CREATE TABLE "t" (id INTEGER PRIMARY KEY, a TEXT, b TEXT)


## 17. Diffing two schemas with sqlite_master
- `ATTACH` a second database and compare `sqlite_master` of both sides with `EXCEPT` or a join.
- Three questions: which objects exist only on one side, which objects exist on both with different `CREATE` text, and which columns differ.
- `pragma_table_info(table, schema)` takes the schema name as its second argument, so one query can read columns of either side.
- Handy for checking "does my migrated dev database match the expected target".

In [22]:
%%sql
ATTACH ':memory:' AS v1;
ATTACH ':memory:' AS v2;
CREATE TABLE v1.users (id INTEGER PRIMARY KEY, email TEXT NOT NULL);
CREATE TABLE v1.legacy_log (id INTEGER PRIMARY KEY, msg TEXT);
CREATE TABLE v2.users (id INTEGER PRIMARY KEY, email TEXT NOT NULL, name TEXT);
CREATE TABLE v2.plans (id INTEGER PRIMARY KEY, label TEXT);
CREATE INDEX v2.idx_users_email ON users(email);
SELECT 'only in v1' AS side, type, name FROM v1.sqlite_master
  WHERE name NOT IN (SELECT name FROM v2.sqlite_master)
UNION ALL
SELECT 'only in v2', type, name FROM v2.sqlite_master
  WHERE name NOT IN (SELECT name FROM v1.sqlite_master)
ORDER BY 1, 3;
SELECT a.name AS object, 'sql text differs' AS finding
FROM v1.sqlite_master a JOIN v2.sqlite_master b ON a.name = b.name
WHERE a.sql <> b.sql;
SELECT 'added in v2' AS change, c.name AS column_name, c.type
FROM pragma_table_info('users', 'v2') c
WHERE c.name NOT IN (SELECT name FROM pragma_table_info('users', 'v1'))
UNION ALL
SELECT 'removed in v2', c.name, c.type
FROM pragma_table_info('users', 'v1') c
WHERE c.name NOT IN (SELECT name FROM pragma_table_info('users', 'v2'))
ORDER BY 1, 2;
DETACH v1;
DETACH v2;

side       | type  | name           
-----------+-------+----------------
only in v1 | table | legacy_log     
only in v2 | index | idx_users_email
only in v2 | table | plans          
(3 rows)

object | finding         
-------+-----------------
users  | sql text differs
(1 row)

change      | column_name | type
------------+-------------+-----
added in v2 | name        | TEXT
(1 row)



## 18. Rollback strategies
- Strategy 1, transaction: DDL in SQLite is transactional. `BEGIN; ... ROLLBACK;` undoes columns, tables and indexes (below).
- Strategy 2, backup first: for risky migrations take a copy (`sqlite3` backup API, `VACUUM INTO 'file'`, or copy the file while nothing writes). Restoring the copy is the real "down" migration.
- Strategy 3, down scripts: write a reverse step for each migration. They are only trustworthy for simple additive changes; dropping data cannot be reversed by a script.
- Strategy 4, forward-only: never go back, fix with a new migration. Many teams pick this one plus good backups.
- A transaction only protects you until `COMMIT`. After that it is a backup or a new forward migration.

In [23]:
%%sql
BEGIN;
ALTER TABLE customers ADD COLUMN temp_flag INTEGER;
CREATE TABLE scratch_during_migration (x);
SELECT count(*) AS has_temp_flag FROM pragma_table_info('customers') WHERE name = 'temp_flag';
ROLLBACK;
SELECT count(*) AS has_temp_flag FROM pragma_table_info('customers') WHERE name = 'temp_flag';
SELECT count(*) AS scratch_exists FROM sqlite_master WHERE name = 'scratch_during_migration';

has_temp_flag
-------------
1            
(1 row)

has_temp_flag
-------------
0            
(1 row)

scratch_exists
--------------
0             
(1 row)



In [24]:
con.execute("CREATE TABLE precious (id INTEGER PRIMARY KEY, note TEXT)")
con.executemany("INSERT INTO precious VALUES (?, ?)", [(1, "a"), (2, "b"), (3, "c")])
snapshot = sqlite3.connect(":memory:")
con.backup(snapshot)                                   # backup API: consistent copy
con.execute("DROP TABLE precious")                      # the 'migration' that went wrong
print("after the bad migration, precious exists?",
      con.execute("SELECT count(*) FROM sqlite_master WHERE name = 'precious'").fetchone()[0] == 1)
snapshot.backup(con)                                    # restore = copy the snapshot back
print("after restore, rows:", con.execute("SELECT count(*) FROM precious").fetchone()[0])
_ = con.execute("DROP TABLE precious")

after the bad migration, precious exists? False
after restore, rows: 3


## 19. Data migration: splitting a column
- Schema changes and data changes should be separate, ordered steps: add columns, copy/transform data, verify, then enforce.
- Here `full_name` becomes `first_name` + `last_name` using `instr` and `substr`.
- Edge cases decide quality: one-word names, several words. The rule used: first word is the first name, everything after the first space is the last name.
- Always finish with a verification query: rebuild the original from the parts and compare.

In [25]:
%%sql
CREATE TABLE people (id INTEGER PRIMARY KEY, full_name TEXT NOT NULL);
INSERT INTO people VALUES (1,'Asha Rao'),(2,'Ben Carter'),(3,'Madonna'),(4,'Mary Ann Smith');
ALTER TABLE people ADD COLUMN first_name TEXT;
ALTER TABLE people ADD COLUMN last_name TEXT;
UPDATE people SET
    first_name = CASE WHEN instr(full_name, ' ') > 0 THEN substr(full_name, 1, instr(full_name, ' ') - 1) ELSE full_name END,
    last_name  = CASE WHEN instr(full_name, ' ') > 0 THEN substr(full_name, instr(full_name, ' ') + 1) ELSE '' END;
SELECT id, full_name, first_name, last_name FROM people ORDER BY id;
SELECT count(*) AS mismatches
FROM people
WHERE trim(first_name || ' ' || last_name) <> full_name;

id | full_name      | first_name | last_name
---+----------------+------------+----------
1  | Asha Rao       | Asha       | Rao      
2  | Ben Carter     | Ben        | Carter   
3  | Madonna        | Madonna    |          
4  | Mary Ann Smith | Mary       | Ann Smith
(4 rows)

mismatches
----------
0         
(1 row)



## 20. Data migration: merging two tables
- Two systems each have their own contact list. The new schema has one `contacts` table keyed by lower-cased email.
- Rules written down first: same email (ignoring case) means same person, and system A's row wins when both systems have that person.
- `ROW_NUMBER() ... PARTITION BY lower(email)` picks one row per person; see 03-Window-Functions/01-Window-Functions.
- Keep a mapping table (old system, old id, new id) so other tables can be repointed afterwards.

In [26]:
%%sql
CREATE TABLE contacts_a (id INTEGER PRIMARY KEY, name TEXT, email TEXT);
CREATE TABLE contacts_b (id INTEGER PRIMARY KEY, name TEXT, email TEXT);
INSERT INTO contacts_a VALUES (1,'Asha Rao','Asha@Example.com'),(2,'Ben Carter','ben@example.com');
INSERT INTO contacts_b VALUES (1,'A. Rao','asha@example.com'),(2,'Chen Wei','chen@example.com');
CREATE TABLE contacts (contact_id INTEGER PRIMARY KEY, email TEXT NOT NULL UNIQUE, name TEXT);
CREATE TABLE contact_map (source TEXT, old_id INTEGER, contact_id INTEGER NOT NULL REFERENCES contacts, PRIMARY KEY (source, old_id));
INSERT INTO contacts (email, name)
SELECT lower(email), name FROM (
    SELECT email, name, row_number() OVER (PARTITION BY lower(email) ORDER BY src, id) AS rn
    FROM (SELECT 1 AS src, id, name, email FROM contacts_a UNION ALL SELECT 2, id, name, email FROM contacts_b)
) WHERE rn = 1
ORDER BY lower(email);
INSERT INTO contact_map
SELECT 'A', a.id, c.contact_id FROM contacts_a a JOIN contacts c ON c.email = lower(a.email)
UNION ALL
SELECT 'B', b.id, c.contact_id FROM contacts_b b JOIN contacts c ON c.email = lower(b.email);
SELECT contact_id, email, name FROM contacts ORDER BY contact_id;
SELECT source, old_id, contact_id FROM contact_map ORDER BY source, old_id;

contact_id | email            | name      
-----------+------------------+-----------
1          | asha@example.com | Asha Rao  
2          | ben@example.com  | Ben Carter
3          | chen@example.com | Chen Wei  
(3 rows)

source | old_id | contact_id
-------+--------+-----------
A      | 1      | 1         
A      | 2      | 2         
B      | 1      | 1         
B      | 2      | 3         
(4 rows)



## 21. Zero-downtime thinking: small batches
- SQLite allows one writer at a time. A huge `UPDATE` holds the write lock for its whole duration and blocks every other writer.
- So backfill big tables in small batches, each its own short transaction, and let other writers in between.
- Pick rows with a stable cursor (`WHERE target IS NULL ORDER BY id LIMIT n`); the loop ends when a batch touches 0 rows.
- Checklist: additive change first, backfill in batches, deploy code that uses the new shape, then contract later.
- Below, 1000 rows are filled in batches of 300: 4 batches (300, 300, 300, 100), a fifth finds nothing.

In [27]:
%%sql
CREATE TABLE events (id INTEGER PRIMARY KEY, kind TEXT NOT NULL);
WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 1000)
INSERT INTO events SELECT i, CASE i % 3 WHEN 0 THEN 'click' WHEN 1 THEN 'view' ELSE 'buy' END FROM n;
ALTER TABLE events ADD COLUMN kind_code INTEGER;

In [28]:
batches = []
while True:
    con.execute("BEGIN")
    cur = con.execute("""
        UPDATE events SET kind_code = CASE kind WHEN 'view' THEN 1 WHEN 'click' THEN 2 ELSE 3 END
        WHERE id IN (SELECT id FROM events WHERE kind_code IS NULL ORDER BY id LIMIT 300)""")
    n = cur.rowcount
    con.execute("COMMIT")
    batches.append(n)
    if n == 0:
        break
print("rows per batch:", batches)
print("rows still NULL:", con.execute("SELECT count(*) FROM events WHERE kind_code IS NULL").fetchone()[0])

rows per batch: [300, 300, 300, 100, 0]
rows still NULL: 0


## 22. Testing migrations
- Test three things: (1) from scratch, all migrations produce the expected structure; (2) data written at an old version survives the upgrade; (3) running the runner twice changes nothing.
- Compare against a hand-written "golden" structure with `pragma_table_info`, not with `sqlite_master.sql` text (section 16).
- Keep one fixture per interesting old version: build the database at version N, insert awkward rows, upgrade, assert.
- Run migrations on a copy of real production data (or a scrubbed sample) before the release.

In [29]:
def structure(db):
    out = []
    for (tbl,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' AND name <> 'schema_migrations' ORDER BY name").fetchall():
        for r in db.execute("SELECT name, type, \"notnull\", pk FROM pragma_table_info(?) ORDER BY cid", (tbl,)):
            out.append((tbl,) + r)
    return out

# 1. from scratch gives the golden structure
t1 = sqlite3.connect(":memory:", isolation_level=None)
migrate(t1, MIGRATIONS)
golden = [("users", "id", "INTEGER", 0, 1), ("users", "email", "TEXT", 1, 0), ("users", "name", "TEXT", 0, 0)]
assert structure(t1) == golden, structure(t1)

# 2. data written at v2 survives the upgrade to v3 (and gets backfilled)
t2 = sqlite3.connect(":memory:", isolation_level=None)
migrate(t2, MIGRATIONS, target=2)
t2.execute("INSERT INTO users (email) VALUES ('zoe@example.com')")
migrate(t2, MIGRATIONS)
assert t2.execute("SELECT name FROM users").fetchall() == [("zoe",)]

# 3. idempotent
assert migrate(t2, MIGRATIONS) == []
assert t2.execute("PRAGMA user_version").fetchone()[0] == 3
print("all 3 migration tests passed")

all 3 migration tests passed
